# Исследование надёжности заёмщиков

Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.

Результаты исследования будут учтены при построении модели **кредитного скоринга** — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

## Шаг 1. Откройте файл с данными и изучите общую информацию

In [551]:
import pandas as pd
from IPython.display import display

data = pd.read_csv('data.csv')
display(data.head())
data.info()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


<a id='general_info'>**Вывод**</a>

В таблице 12 столбцов и 21525 строчел. В каждой строчки таблицы - данные о клиенте и его кредите.

Согласно документации к данным:
* `children` — количество детей в семье;
* `days_employed` — общий трудовой стаж в днях;
* `dob_years` — возраст клиента в годах;
* `education` — уровень образования клиента;
* `education_id` — идентификатор уровня образования;
* `family_status` — семейное положение;
* `family_status_id` — идентификатор семейного положения;
* `gender` — пол клиента;
* `income_type` — тип занятости;
* `debt` — имел ли задолженность по возврату кредита;
* `total_income` — ежемесячный доход;
* `purpose` — цель получения кредита.

В столбцах `total_income` и `days_employed` встречаются пропуски. Также тип в этих столбцах `float`, хотя по природе данных (количество дней и доход) должен быть `int`. Типы данных других столбцов отображают действительность.
В `days_emploted` можно увидеть много отрицательных значений, это также артефакт.

## Шаг 2. Предобработка данных

### Обработка пропусков

Сначала посчитаем количество пропущенных значений в таблице.

In [518]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

Как видим, количество пропущенных значений в столбцах `days_employed` и `total_income` одинаково, и встречаются пропуски у одних и тех же клиентов. Скорее всего, у источника информации для этих клиентов просто не было данных про стаж и доход.
Столбец `days_employed` никак не влияет на проверку гипотез и конечный результат, поэтому мы можем его удалить или заполнить пропуски произвольным значением, например нулем.

In [519]:
data['days_employed'] = data['days_employed'].fillna(0)

Но пропуски в `total_income` могут исказить конечный результат. Лучшим вариантом устранить пропуски было бы уточнить информацию у разработчика, который дал нам эти данные, но такой возможности у нас нет. Поэтому чтобы заполнить пропущенные значения, мы посчитаем медиану для каждой группы по типу занятости и запишем соответствующие значения для каждого клиента, относительно его `income_type`.

In [520]:
data['total_income'] = data.groupby(['income_type'])['total_income'].apply(
    lambda income: income.fillna(income.median())
)

Убедимся, что в таблице не осталось пропусков

In [521]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Замена типа данных

Как мы уже выяснили ранее, столбцы `total_income` и `days_employed` распознаны как `float`. Это означает, что в столбцах есть дробные значения, что противоречит природе данных. Мы не можем точно знать причину такого типа, но скорее всего это произошло в результате деления.

Например, для `days_employed` могли произойти следующие вычисления:
`secs_employed = end_of_work - start_of_work`, где `end_of_work` и `start_of_work` время конца и начало работы в формате **unix time**
`days_employed = secs_employed / 60 / 60 / 24` - преобразование числа из секунд в дни

Поэтому для более удобной работы в дальнейшем, превратим данные `total_income`, `days_employed` в целочисленные с помощью метода `astype()`.

In [522]:
data['total_income'] = data['total_income'].astype('int')
data['days_employed'] = data['days_employed'].astype('int')

Проверим полученный результат, вызвав метод `info()`

In [523]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   children          21525 non-null  int64 
 1   days_employed     21525 non-null  int64 
 2   dob_years         21525 non-null  int64 
 3   education         21525 non-null  object
 4   education_id      21525 non-null  int64 
 5   family_status     21525 non-null  object
 6   family_status_id  21525 non-null  int64 
 7   gender            21525 non-null  object
 8   income_type       21525 non-null  object
 9   debt              21525 non-null  int64 
 10  total_income      21525 non-null  int64 
 11  purpose           21525 non-null  object
dtypes: int64(7), object(5)
memory usage: 2.0+ MB


Видно, что в столбцах `total_income` и `days_employed` сохраняются только целочисленные значения.

Когда мы узучали [общую информацию](#general_info) о таблице, то заметили, что в столбце много отрицательных значений. Можем предположить, что они возникли в результате неправильной перестановки операндов при вычитании (`start_of_work - end_of_work`). Для того чтобы данные отражали действительность, превратим отрицательные значения в положительные с помощью функции `abs()`.

In [524]:
data['days_employed'] = abs(data['days_employed'])

### Обработка дубликатов

In [525]:
data.duplicated().sum()

54

В таблице у нас 54 дубликата. Маловероятно, что мы можем встретить разных людей, у которых полностью совпадают все значения в столбцах. Поэтому будет считать, что дубликаты возникли в результате сбоя в базе данных и мы можем избавиться от них с помощью метода `drop_duplicates()`. Удалим явные дубликаты с удалением старых индексов и формированием новых:

In [526]:
data = data.drop_duplicates().reset_index(drop=True)

Убедимся, что мы полностью избавились от явных дубликатов

In [527]:
data.duplicated().sum()

0

Теперь попробуем отыскать неявные дубликаты в каждом столбце, где их наличие вероятно:

In [528]:
print(
    data['education'].value_counts(),
    data['family_status'].value_counts(),
    data['income_type'].value_counts(),
    data['purpose'].value_counts(),
    sep='\n\n'
)

среднее                13705
высшее                  4710
СРЕДНЕЕ                  772
Среднее                  711
неоконченное высшее      668
ВЫСШЕЕ                   273
Высшее                   268
начальное                250
Неоконченное высшее       47
НЕОКОНЧЕННОЕ ВЫСШЕЕ       29
НАЧАЛЬНОЕ                 17
Начальное                 15
ученая степень             4
Ученая степень             1
УЧЕНАЯ СТЕПЕНЬ             1
Name: education, dtype: int64

женат / замужем          12344
гражданский брак          4163
Не женат / не замужем     2810
в разводе                 1195
вдовец / вдова             959
Name: family_status, dtype: int64

сотрудник          11091
компаньон           5080
пенсионер           3837
госслужащий         1457
безработный            2
предприниматель        2
студент                1
в декрете              1
Name: income_type, dtype: int64

свадьба                                   793
на проведение свадьбы                     773
сыграть свадьбу    

Видно, что в столбце `education` много неявных дубликатов, отличающихся регистром. А в столбце `pursope` одни и те же цели получения кредита записаны в разных формах. Во всех остальных столбцах значения уникальны.
Чтобы избавиться от дубликатов в `education` приведем все значения к нижнему регистру с помощью метода `str.lower()`. Избавление от дубликатов в `pursope` требует более сложного процесса – [**лемматизации**](#lemmatisation).

In [529]:
data['education'] = data['education'].str.lower()

Проверим, появились ли дубликаты в таблице после наших преобразований.

In [530]:
data.duplicated().sum()

17

С поможу метода `drop_duplicates()` избавимся от дубликатов. А после этого проверим удалили ли мы их всех.


In [531]:
data = data.drop_duplicates().reset_index(drop=True)
data.duplicated().sum()

0

Дубликатов в таблице нет, можем переходить к следующему шагу – лемматизации.

### <a id='lemmatisation'>Лемматизация</a>

Выведем список уникальных значений, которые есть в столбце `purpose`

In [532]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

Всего 38 разных значений, но если присмотреться, то у многих одна цель, а формулировка - разная .

Например, весь список:
* покупка жилья
* покупка жилья для семьи
* покупка недвижимости
* покупка жилой недвижимости
* строительство собственной недвижимости
* строительство недвижимости

описывает одну цель получения кредита – для недвижимости.

Поэтому мы можем выделить несколько категорий для целей:
* коммерция
* недвижимость
* свадьба
* автомобиль
* образование

Чтобы перейти от разного описания цели к одному консистентному – будем использовать процесс лемматизации.

Для начала импортируем библиотеку `pymystem3` для лемметизации.

In [533]:
from pymystem3 import Mystem
m = Mystem()

Леммотизируем слова для каждой уникальной цели:

In [534]:
for unique_purpose in data['purpose'].unique():
    purpose_lemmas = m.lemmatize(unique_purpose)
    print(purpose_lemmas)

['покупка', ' ', 'жилье', '\n']
['приобретение', ' ', 'автомобиль', '\n']
['дополнительный', ' ', 'образование', '\n']
['сыграть', ' ', 'свадьба', '\n']
['операция', ' ', 'с', ' ', 'жилье', '\n']
['образование', '\n']
['на', ' ', 'проведение', ' ', 'свадьба', '\n']
['покупка', ' ', 'жилье', ' ', 'для', ' ', 'семья', '\n']
['покупка', ' ', 'недвижимость', '\n']
['покупка', ' ', 'коммерческий', ' ', 'недвижимость', '\n']
['покупка', ' ', 'жилой', ' ', 'недвижимость', '\n']
['строительство', ' ', 'собственный', ' ', 'недвижимость', '\n']
['недвижимость', '\n']
['строительство', ' ', 'недвижимость', '\n']
['на', ' ', 'покупка', ' ', 'подержать', ' ', 'автомобиль', '\n']
['на', ' ', 'покупка', ' ', 'свой', ' ', 'автомобиль', '\n']
['операция', ' ', 'с', ' ', 'коммерческий', ' ', 'недвижимость', '\n']
['строительство', ' ', 'жилой', ' ', 'недвижимость', '\n']
['жилье', '\n']
['операция', ' ', 'со', ' ', 'свой', ' ', 'недвижимость', '\n']
['автомобиль', '\n']
['заниматься', ' ', 'образование'

Теперь мы можем сопоставить наличие определенной леммы с категориями. Создадим словарь, где ключом будет лемма, а значением – наша категория.

In [535]:
purpose_categories = {
        'коммерческий': 'коммерция',
        'жилье': 'недвижимость',
        'недвижимость': 'недвижимость',
        'свадьба': 'свадьба',
        'автомобиль': 'автомобиль',
        'образование': 'образование'
    }

Напишем функцию, которая будет принимать значение `purpose` и возвращать соответствующую категорию с нашего словаря. И с помощью метода `apply` применим эту функцию для всего столбца `purpose`, а значение функции запишем в новый столбец `purpose_category`.

In [536]:
def get_purpose_category(purpose):
    lemmas = m.lemmatize(purpose)
    for lemma, category in purpose_categories.items():
        if lemma in lemmas:
            return category

data['purpose_category'] = data['purpose'].apply(get_purpose_category)

**Вывод**
С помощью лемматизации удалось сократить список с 38 целей до 5. В будущем это поможет нам при проверке гипотезы -
**Как разные цели кредита влияют на его возврат в срок?**

Одна из гипотез, которые нам нужно проверить, звучит так - **Есть ли зависимость между наличием детей и возвратом кредита в срок?** Значит, нужно сгруппировать наши данные по значению столбца `children`.
Разделим данные о клиентах на две категории:
* если значение `children` больше нуля, значит **есть дети**
* если значение `children` равно нулю - семья **без детей**

Запишем правила классификации клиентов как функцию, на вход которой попадает количество детей, а возвращает она категорию клиента.

In [537]:
def family_children_type(children_amount):
    if children_amount == 0:
        return 'без детей'
    else:
        return 'есть дети'

Протестируем работу функции для каждого правила:

In [538]:
print(family_children_type(0))
print(family_children_type(2))

без детей
есть дети


Написанная функция работает корректно. Осталось создать отдельный столбец с категорией, и в его ячейках записать значения возвращаемые функцией. Для этого будем использовать метод `apply()`

In [539]:
data['children_category'] = data['children'].apply(family_children_type)
display(data.head())

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,purpose_category,children_category
0,1,8437,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,недвижимость,есть дети
1,1,4024,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,автомобиль,есть дети
2,0,5623,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,недвижимость,без детей
3,3,4124,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,образование,есть дети
4,0,340266,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,свадьба,без детей


Данные категоризированы по наличию детей у клиента, и можно также категоризировать еще данные по ежемесячному доходу. Так как у нас нет опорных точек, с помощью которых мы могли бы поделить данные на категории бедный, средний и богатый класс, то разделим данные с помощью процентилей на три группы. Найдем необходимые точки:

In [540]:
data['total_income'].describe(percentiles=[0.33, 0.66])

count    2.145400e+04
mean     1.653196e+05
std      9.818730e+04
min      2.066700e+04
33%      1.185140e+05
50%      1.425940e+05
66%      1.723570e+05
max      2.265604e+06
Name: total_income, dtype: float64

Видно, как мы можем разделить данные на три категории с помощью процентилей:
* **Q1** - `total_income < 118514`
* **Q2** - `118514 <= total_income < 172357`
* **Q3** - `total_income >= 172357`

Теперь мы можем записать функцию, которая будет классифицировать клиентов по `total_income`. На вход функция принимает количество ежемесячного дохода и возвращает категорию.

In [541]:
def income_category(income):
    if income < 118514:
        return 'Q1'
    elif 118514 <= income < 172357:
        return 'Q2'
    else:
        return 'Q3'

Проверим, работает ли функция правильно:

In [542]:
print(income_category(250))
print(income_category(125550))
print(income_category(345550))

Q1
Q2
Q3


Написанная функция работает корректно, теперь мы можем применить ее ко всем данным:

In [543]:
data['income_category'] = data['total_income'].apply(income_category)
data.head()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,purpose_category,children_category,income_category
0,1,8437,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,недвижимость,есть дети,Q3
1,1,4024,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,автомобиль,есть дети,Q1
2,0,5623,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,недвижимость,без детей,Q2
3,3,4124,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,образование,есть дети,Q3
4,0,340266,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,свадьба,без детей,Q2


**Вывод**
Мы смогли классифицировать данные по наличию детей у клиента, поделив данные на две группы – **"есть дети"** и **"нет детей"**. Также мы классифицировали клиентов по ежемесячному доходу, поделив их на три группы: **Q1** - 33% зарабатывающих меньше всего, **Q3** - 33% зарабатывающих больше всего и **Q2** - все остальные. Эти манипуляции с данными необходимы для того, чтобы при проверке гипотез мы смогли сделать статистически значимые выводы.

## Шаг 3. Ответьте на вопросы

- Есть ли зависимость между наличием детей и возвратом кредита в срок?

Чтобы ответить на этот вопрос, необходимо сгруппировать данные по `children_category`. В каждой категории посчитать процент клиентов, у которых был долг по кредиту.

In [544]:
data_by_children = data.groupby('children_category')
print(data_by_children['debt'].sum() / data_by_children['children'].count())

children_category
без детей    0.075438
есть дети    0.092082
dtype: float64


**Вывод**
Клиенты без детей возвращают кредиты примерно на 20% чаще.

- Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [545]:
data_by_family_status = data.groupby('family_status')
print(data['debt'].sum() / data['family_status'].count())
print(data_by_family_status['debt'].sum() / data_by_family_status['family_status'].count())

0.08115036822970076
family_status
Не женат / не замужем    0.097509
в разводе                0.071130
вдовец / вдова           0.065693
гражданский брак         0.093471
женат / замужем          0.075452
dtype: float64


**Вывод**
Процент невыплаты колеблется в зависимости от семейного статуса в пределах 6.56-9.75%. Согласно данным, найболее надежные клиенты - вдовцы и вдовы, а найменее - не состоящие в браке.

- Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [546]:
data_by_income = data.groupby('income_category')
print(data_by_income['debt'].sum() / data_by_income['income_category'].count())

income_category
Q1    0.081085
Q2    0.088431
Q3    0.074212
dtype: float64


**Вывод**
Разница есть, но она не такая значительная как в предыдущих пунктах. Однако можем сказать, что найболее ответственные заёмщики - лица с высоким уровнем дохода.

- Как разные цели кредита влияют на его возврат в срок?

In [547]:
data_by_purpose = data.groupby('purpose_category')
print(data_by_purpose['debt'].sum() / data_by_purpose['purpose'].count())

purpose_category
автомобиль      0.093590
коммерция       0.075515
недвижимость    0.071895
образование     0.092200
свадьба         0.080034
dtype: float64


**Вывод**
Цели кредита влияют на его выплату. Клиенты, которые берут кредит на образование, ожидаемо имеют больше задолжености, так как вероятно еще не имеют постояного дохода. Также кредитование на потребительские цели (автомобиль, свадьба) говорит о низком уровне дохода и худшей способности выплачивать долги, в то время как инвестиционные цели (коммерция и недвижимость) коррелируют с лучшими выплатами.

## Шаг 4. Общий вывод

Мы проверили 4 гипотезы и установили:
1. Клиенты без детей возвращают кредиты чаще чем те, у которых дети есть.
2. Заёмщики, не состоящие в браке, хуже всего возвращают кредит, в то время как вдовцы и вдовы зарекомендовали себя как должники, которым можно доверять.
3. Как и ожидалось, наиболее ответственными в категории по месячному доходу оказались лица с высоким уровнем заработка.
4. А инвестиционные цели (коммерция и недвижимость) коррелируют с лучшими выплатами.

Процент проблемных должников колеблется в пределах 6.6-9.8% в зависимости от группы, на всех данных этот процент составляет 8%. Мне кажется, что мы не можем делать какие-либо выводы, просто посмотрев на эти цифры, как минимум нам необходимо понимать с чем их сравнивать. И для того чтобы утверждать о хотя бы какой-то зависимости между выплатами и определенной группой, нужно провести статистический анализ, о котором мне еще очень мало известно.